In [1]:
# Cell 1 — Setup, imports, Google Drive

import os
import gc
import json
import math
import time
import random
import hashlib
import shutil
import zipfile
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Iterable

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Không chạy trong Google Colab; bỏ qua bước mount Drive.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB")


Mounted at /content/drive
Device: cuda | PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB


In [2]:
# Cell 2 — Step 6 configuration

STEP5_EXPERIMENT_TAG = "conference_step5_literature_subset_v1"
STEP5_PHASE = "confirmation"
STEP6_EXPERIMENT_TAG = "conference_step6_public_baselines_v1"
PROTOCOL_NAME = "exact_feature_group"
BENCHMARK_SPLIT_SEED = 2027

DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/THAT_IDS_RERUN")
STEP5_ROOT = DRIVE_RUN_ROOT / STEP5_EXPERIMENT_TAG
STEP5_CONFIRMATION_ROOT = STEP5_ROOT / STEP5_PHASE
STEP5_SUBSET_DEFINITION_ROOT = STEP5_ROOT / "subset_definition"
OUTPUT_ROOT = DRIVE_RUN_ROOT / STEP6_EXPERIMENT_TAG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = {
    "UNSW_V2": [
        "/content/drive/MyDrive/datasets/NF-UNSW-NB15-v2.csv",
        "/content/drive/MyDrive/NF-UNSW-NB15-v2.csv",
    ],
    "CIC_V2": [
        "/content/drive/MyDrive/datasets/NF-CSE-CIC-IDS2018-v2.csv",
        "/content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv",
    ],
}

# Chạy một dataset mỗi lần nếu muốn giảm rủi ro mất runtime.
DATASETS_TO_RUN = ["CIC_V2", "UNSW_V2"]
PUBLIC_BASELINES_TO_RUN = ["egraphsage_public", "graphids_public"]
MODEL_SEEDS = [42, 123, 456]

# Shared controlled graph batches. The rows/split/scaler are frozen by Step 5.
PUBLIC_GRAPH_WINDOW_SIZE = 5000
CSV_CHUNK_SIZE = 500_000
CLIP_MIN = -10.0
CLIP_MAX = 10.0
SAVE_LAST_EVERY = 1
GRAD_CLIP_NORM = 1.0
STRICT_COMPLETENESS = False

# E-GraphSAGE official binary notebook uses two edge-aware layers, hidden=128,
# dropout=0.2, balanced cross entropy and Adam with its default 1e-3 LR.
EGRAPH_CONFIG = {
    "hidden_dim": 128,
    "dropout": 0.20,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "max_epochs": 100,
    "patience": 20,
    "node_definition": "ip_port",  # official notebook joins IP and L4 port
    "selection_metric": "val_f1_macro",
}

# GraphIDS main-branch public configs for v2 datasets. Architecture/training objective
# follows the official PyG implementation; only the frozen Step-5 loader is substituted.
GRAPHIDS_CONFIG = {
    "UNSW_V2": {
        "edim_out": 72,
        "learning_rate": 1.1e-5,
        "encoder_weight_decay": 0.6,
        "ae_weight_decay": 0.04573136485742748,
        "dropout": 0.75,
        "ae_embedding_dim": 48,
        "ae_dropout": 0.0,
        "ae_feedforward_dim": 256,
        "num_heads": 4,
        "num_layers": 1,
        "sequence_window": 512,
        "sequence_step": 512,
        "ae_batch_size": 64,
        "mask_ratio": 0.15,
        "positional_encoding": "None",
        "max_epochs": 100,
        "patience": 20,
        "selection_metric": "val_pr_auc",
        "node_definition": "ip",
    },
    "CIC_V2": {
        "edim_out": 64,
        "learning_rate": 7.423525164428174e-5,
        "encoder_weight_decay": 0.6,
        "ae_weight_decay": 0.011263928281120002,
        "dropout": 0.50,
        "ae_embedding_dim": 32,
        "ae_dropout": 0.20,
        "ae_feedforward_dim": 256,
        "num_heads": 4,
        "num_layers": 1,
        "sequence_window": 512,
        "sequence_step": 512,
        "ae_batch_size": 64,
        "mask_ratio": 0.15,
        "positional_encoding": "None",
        "max_epochs": 100,
        "patience": 20,
        "selection_metric": "val_pr_auc",
        "node_definition": "ip",
    },
}

PUBLIC_SOURCE_METADATA = {
    "egraphsage_public": {
        "paper": "E-GraphSAGE: A Graph Neural Network based Intrusion Detection System for IoT",
        "venue": "NOMS 2022",
        "repository": "https://github.com/waimorris/E-GraphSAGE",
        "implementation_scope": "controlled reimplementation of the official binary mean-aggregation architecture",
    },
    "graphids_public": {
        "paper": "Self-Supervised Learning of Graph Representations for Network Intrusion Detection",
        "venue": "NeurIPS 2025",
        "repository": "https://github.com/lorenzo9uerra/GraphIDS",
        "implementation_scope": "controlled reimplementation of the official PyG architecture and benign-only objective",
    },
}

# Test-curve summaries. These are curve statistics, not deployment thresholds selected on test.
TEST_CURVE_FPR_TARGETS = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
TEST_CURVE_RECALL_TARGETS = [0.90, 0.95, 0.99]

print("Step 5 confirmation root:", STEP5_CONFIRMATION_ROOT)
print("Step 6 output root:", OUTPUT_ROOT)
print("Datasets:", DATASETS_TO_RUN)
print("Public baselines:", PUBLIC_BASELINES_TO_RUN)
print("Model seeds:", MODEL_SEEDS)
print("Expected public-baseline tasks:", len(DATASETS_TO_RUN) * len(PUBLIC_BASELINES_TO_RUN) * len(MODEL_SEEDS))


Step 5 confirmation root: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step5_literature_subset_v1/confirmation
Step 6 output root: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step6_public_baselines_v1
Datasets: ['CIC_V2', 'UNSW_V2']
Public baselines: ['egraphsage_public', 'graphids_public']
Model seeds: [42, 123, 456]
Expected public-baseline tasks: 12


In [3]:
# Cell 3 — Reproducibility, checkpoint, and path helpers

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def capture_rng_state() -> Dict[str, Any]:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def _cpu_byte_tensor(value: Any) -> torch.Tensor:
    if isinstance(value, torch.Tensor):
        return value.detach().to(device="cpu", dtype=torch.uint8).contiguous()
    return torch.as_tensor(value, dtype=torch.uint8, device="cpu").contiguous()


def restore_rng_state(state: Optional[Dict[str, Any]]) -> None:
    if not state:
        return
    if "python" in state:
        random.setstate(state["python"])
    if "numpy" in state:
        np.random.set_state(state["numpy"])
    if "torch" in state:
        torch.set_rng_state(_cpu_byte_tensor(state["torch"]))
    if torch.cuda.is_available() and "cuda" in state:
        cuda_states = state["cuda"]
        if isinstance(cuda_states, torch.Tensor):
            cuda_states = [cuda_states]
        for device_index, cuda_state in enumerate(list(cuda_states)[: torch.cuda.device_count()]):
            torch.cuda.set_rng_state(_cpu_byte_tensor(cuda_state), device=device_index)


def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def atomic_torch_save(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def load_torch_checkpoint(path: Path, map_location: torch.device) -> Dict[str, Any]:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_history(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    return pd.read_csv(path).to_dict("records")


def save_history(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).sort_values("epoch").to_csv(path, index=False)


def resolve_dataset_path(dataset_name: str) -> str:
    for candidate in DATA_PATH_CANDIDATES.get(dataset_name, []):
        if Path(candidate).exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy dataset " + dataset_name + ". Đã thử:\n" +
        "\n".join(DATA_PATH_CANDIDATES.get(dataset_name, []))
    )


def step5_dataset_dir(dataset_name: str) -> Path:
    return STEP5_CONFIRMATION_ROOT / dataset_name


def step5_shared_dir(dataset_name: str) -> Path:
    return (
        step5_dataset_dir(dataset_name)
        / PROTOCOL_NAME
        / "shared"
        / f"seed_{BENCHMARK_SPLIT_SEED}"
    )


def subset_definition_dir(dataset_name: str) -> Path:
    return STEP5_SUBSET_DEFINITION_ROOT / dataset_name


def task_fingerprint(
    dataset_name: str,
    baseline_name: str,
    seed: int,
    feature_names: List[str],
    subset_fingerprint: str,
    split_fingerprint: str,
    config: Dict[str, Any],
) -> str:
    payload = {
        "step6_tag": STEP6_EXPERIMENT_TAG,
        "step5_tag": STEP5_EXPERIMENT_TAG,
        "dataset": dataset_name,
        "baseline": baseline_name,
        "seed": int(seed),
        "feature_names": list(feature_names),
        "subset_fingerprint": subset_fingerprint,
        "split_fingerprint": split_fingerprint,
        "graph_window_size": PUBLIC_GRAPH_WINDOW_SIZE,
        "config": config,
        "source": PUBLIC_SOURCE_METADATA[baseline_name],
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:20]


In [4]:
# Cell 4 — Load the exact fixed Step-5 subset, split, and scaler

@dataclass
class FixedSubsetBundle:
    name: str
    feature_names: List[str]
    X_raw: np.ndarray
    src_ip: np.ndarray
    dst_ip: np.ndarray
    src_port: np.ndarray
    dst_port: np.ndarray
    protocol: np.ndarray
    labels: np.ndarray
    attack_codes: np.ndarray
    attack_classes: np.ndarray
    subset_global_indices: np.ndarray
    subset_fingerprint: str


def read_step5_feature_names(dataset_name: str) -> List[str]:
    feature_path = step5_dataset_dir(dataset_name) / "model_feature_names.csv"
    if not feature_path.exists():
        raise FileNotFoundError(feature_path)
    table = pd.read_csv(feature_path)
    if "feature" not in table.columns:
        raise ValueError(f"Unexpected feature file schema: {feature_path}")
    return table.sort_values("feature_index")["feature"].astype(str).tolist()


def load_fixed_subset_bundle(dataset_name: str) -> FixedSubsetBundle:
    csv_path = Path(resolve_dataset_path(dataset_name))
    definition_dir = subset_definition_dir(dataset_name)
    subset_idx_path = definition_dir / "subset_indices_int32.npy"
    subset_manifest_path = definition_dir / "subset_manifest.json"

    if not subset_idx_path.exists() or not subset_manifest_path.exists():
        raise FileNotFoundError(
            f"Missing Step-5 subset definition for {dataset_name}: {definition_dir}"
        )

    subset_manifest = load_json(subset_manifest_path)
    subset_idx = np.asarray(np.load(subset_idx_path, mmap_mode="r"), dtype=np.int64)
    if np.any(np.diff(subset_idx) < 0):
        raise RuntimeError("Step-5 subset indices are not sorted; chunk loader would change row order.")

    feature_names = read_step5_feature_names(dataset_name)
    required_context = [
        "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "L4_SRC_PORT", "L4_DST_PORT",
        "PROTOCOL", "Label", "Attack",
    ]
    usecols = list(dict.fromkeys(feature_names + required_context))

    print(f"\nLoading fixed Step-5 subset for {dataset_name}")
    print(f"CSV: {csv_path}")
    print(f"Subset rows: {len(subset_idx):,} | features: {len(feature_names)}")

    feature_chunks: List[np.ndarray] = []
    context_chunks: Dict[str, List[np.ndarray]] = {name: [] for name in required_context}
    cursor = 0
    global_start = 0
    t0 = time.time()

    for chunk_no, df_chunk in enumerate(
        pd.read_csv(csv_path, usecols=usecols, chunksize=CSV_CHUNK_SIZE, low_memory=False),
        start=1,
    ):
        global_end = global_start + len(df_chunk)
        left = np.searchsorted(subset_idx, global_start, side="left")
        right = np.searchsorted(subset_idx, global_end, side="left")

        if right > left:
            local_rows = subset_idx[left:right] - global_start
            selected = df_chunk.iloc[local_rows]

            x = np.empty((len(selected), len(feature_names)), dtype=np.float32)
            for j, col in enumerate(feature_names):
                arr = pd.to_numeric(selected[col], errors="coerce").to_numpy(
                    dtype=np.float32, copy=False
                )
                x[:, j] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            feature_chunks.append(x)

            for col in required_context:
                context_chunks[col].append(selected[col].to_numpy(copy=True))
            cursor += len(selected)

        global_start = global_end
        if chunk_no % 10 == 0 or cursor == len(subset_idx):
            print(f"  CSV chunks={chunk_no} | selected={cursor:,}/{len(subset_idx):,}")
        del df_chunk
        if cursor == len(subset_idx):
            break

    if cursor != len(subset_idx):
        raise RuntimeError(
            f"Loaded {cursor:,} rows but Step-5 subset contains {len(subset_idx):,}."
        )

    X_raw = np.concatenate(feature_chunks, axis=0)
    context = {k: np.concatenate(v, axis=0) for k, v in context_chunks.items()}
    labels = (
        pd.to_numeric(pd.Series(context["Label"]), errors="coerce")
        .fillna(0)
        .to_numpy(dtype=np.int8)
    )
    if not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError("Label is not binary.")

    attack_codes, attack_classes = pd.factorize(
        pd.Series(context["Attack"]).fillna("<MISSING_ATTACK>").astype(str),
        sort=False,
    )
    attack_codes = attack_codes.astype(np.int16, copy=False)
    attack_classes = np.asarray(attack_classes, dtype=str)

    expected_rows = int(subset_manifest.get("achieved_rows", len(subset_idx)))
    if len(labels) != expected_rows:
        raise RuntimeError(
            f"Subset manifest row mismatch: loaded={len(labels)}, manifest={expected_rows}"
        )

    print(f"Loaded fixed subset in {(time.time() - t0) / 60:.1f} min")
    print("Label counts:", dict(zip(*np.unique(labels, return_counts=True))))

    return FixedSubsetBundle(
        name=dataset_name,
        feature_names=feature_names,
        X_raw=X_raw,
        src_ip=context["IPV4_SRC_ADDR"],
        dst_ip=context["IPV4_DST_ADDR"],
        src_port=context["L4_SRC_PORT"],
        dst_port=context["L4_DST_PORT"],
        protocol=context["PROTOCOL"],
        labels=labels,
        attack_codes=attack_codes,
        attack_classes=attack_classes,
        subset_global_indices=subset_idx,
        subset_fingerprint=str(subset_manifest["subset_fingerprint"]),
    )


def load_frozen_step5_artifacts(
    bundle: FixedSubsetBundle,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Any, Dict[str, Any]]:
    shared_dir = step5_shared_dir(bundle.name)
    split_path = shared_dir / "split_indices.npz"
    split_meta_path = shared_dir / "split_metadata.json"
    scaler_path = shared_dir / "scaler.joblib"
    overlap_path = shared_dir / "split_overlap_audit.json"

    for path in [split_path, split_meta_path, scaler_path, overlap_path]:
        if not path.exists():
            raise FileNotFoundError(path)

    split_data = np.load(split_path)
    train_idx = np.asarray(split_data["train_idx"], dtype=np.int64)
    val_idx = np.asarray(split_data["val_idx"], dtype=np.int64)
    test_idx = np.asarray(split_data["test_idx"], dtype=np.int64)
    split_meta = load_json(split_meta_path)
    overlap = load_json(overlap_path)
    scaler = joblib.load(scaler_path)

    if split_meta.get("protocol_name") != PROTOCOL_NAME:
        raise RuntimeError("Step-5 split protocol mismatch.")
    if max(
        float(overlap.get("validation_feature_overlap_rate", 1.0)),
        float(overlap.get("test_feature_overlap_rate", 1.0)),
    ) != 0.0:
        raise RuntimeError("Frozen Step-5 split has non-zero exact-feature overlap.")
    if list(getattr(scaler, "feature_names_in_", [])) and list(scaler.feature_names_in_) != bundle.feature_names:
        raise RuntimeError("Scaler feature names do not match Step-5 model features.")

    all_idx = np.concatenate([train_idx, val_idx, test_idx])
    if len(np.unique(all_idx)) != len(all_idx):
        raise RuntimeError("Frozen train/validation/test indices overlap by row index.")
    if all_idx.min() < 0 or all_idx.max() >= len(bundle.labels):
        raise RuntimeError("Frozen split index outside the fixed subset.")

    print(
        f"Frozen split: train={len(train_idx):,}, val={len(val_idx):,}, "
        f"test={len(test_idx):,} | fingerprint={split_meta['split_fingerprint']}"
    )
    return train_idx, val_idx, test_idx, scaler, split_meta


In [5]:
# Cell 5 — Fixed public-baseline graph windows

@dataclass
class PublicGraphWindow:
    edge_feat: torch.Tensor
    labels: torch.Tensor
    attack_codes: torch.Tensor
    src_ip: torch.Tensor
    dst_ip: torch.Tensor
    n_ip_nodes: int
    src_endpoint: torch.Tensor
    dst_endpoint: torch.Tensor
    n_endpoint_nodes: int


def _string_array(values: np.ndarray, missing: str) -> np.ndarray:
    return pd.Series(values, copy=False).fillna(missing).astype(str).to_numpy()


def build_public_graph_window(
    bundle: FixedSubsetBundle,
    row_idx: np.ndarray,
    scaler: Any,
) -> PublicGraphWindow:
    scaled = scaler.transform(bundle.X_raw[row_idx])
    scaled = np.clip(scaled, CLIP_MIN, CLIP_MAX).astype(np.float32, copy=False)

    src_ip_values = _string_array(bundle.src_ip[row_idx], "<MISSING_IP>")
    dst_ip_values = _string_array(bundle.dst_ip[row_idx], "<MISSING_IP>")
    combined_ip = np.concatenate([src_ip_values, dst_ip_values])
    ip_codes, _ = pd.factorize(combined_ip, sort=False)
    n = len(row_idx)
    src_ip = ip_codes[:n].astype(np.int64, copy=False)
    dst_ip = ip_codes[n:].astype(np.int64, copy=False)

    src_port_values = _string_array(bundle.src_port[row_idx], "<MISSING_PORT>")
    dst_port_values = _string_array(bundle.dst_port[row_idx], "<MISSING_PORT>")
    src_endpoint_values = np.char.add(np.char.add(src_ip_values, ":"), src_port_values)
    dst_endpoint_values = np.char.add(np.char.add(dst_ip_values, ":"), dst_port_values)
    combined_endpoint = np.concatenate([src_endpoint_values, dst_endpoint_values])
    endpoint_codes, _ = pd.factorize(combined_endpoint, sort=False)
    src_endpoint = endpoint_codes[:n].astype(np.int64, copy=False)
    dst_endpoint = endpoint_codes[n:].astype(np.int64, copy=False)

    return PublicGraphWindow(
        edge_feat=torch.from_numpy(scaled),
        labels=torch.from_numpy(bundle.labels[row_idx].astype(np.int64, copy=False)),
        attack_codes=torch.from_numpy(bundle.attack_codes[row_idx].astype(np.int64, copy=False)),
        src_ip=torch.from_numpy(src_ip),
        dst_ip=torch.from_numpy(dst_ip),
        n_ip_nodes=int(ip_codes.max() + 1) if len(ip_codes) else 0,
        src_endpoint=torch.from_numpy(src_endpoint),
        dst_endpoint=torch.from_numpy(dst_endpoint),
        n_endpoint_nodes=int(endpoint_codes.max() + 1) if len(endpoint_codes) else 0,
    )


def prepare_public_graph_windows(
    bundle: FixedSubsetBundle,
    split_idx: np.ndarray,
    scaler: Any,
    description: str,
) -> List[PublicGraphWindow]:
    windows: List[PublicGraphWindow] = []
    n_windows = math.ceil(len(split_idx) / PUBLIC_GRAPH_WINDOW_SIZE)
    t0 = time.time()
    for window_no in range(n_windows):
        rows = split_idx[
            window_no * PUBLIC_GRAPH_WINDOW_SIZE:
            (window_no + 1) * PUBLIC_GRAPH_WINDOW_SIZE
        ]
        if len(rows) == 0:
            continue
        windows.append(build_public_graph_window(bundle, rows, scaler))
        if (window_no + 1) % 100 == 0 or window_no + 1 == n_windows:
            print(f"  {description}: {window_no + 1}/{n_windows} windows")
    print(
        f"Prepared {description}: {len(windows)} windows in "
        f"{(time.time() - t0) / 60:.1f} min"
    )
    return windows


def select_benign_train_indices(bundle: FixedSubsetBundle, train_idx: np.ndarray) -> np.ndarray:
    benign = train_idx[bundle.labels[train_idx] == 0]
    if len(benign) == 0:
        raise RuntimeError("GraphIDS benign-only training split is empty.")
    print(f"GraphIDS benign-only train rows: {len(benign):,}/{len(train_idx):,}")
    return benign


In [6]:
# Cell 6 — Metrics, validation thresholds, and test-curve operating points

def safe_divide(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


def find_best_threshold_macro_f1_exact(
    scores: np.ndarray,
    labels: np.ndarray,
) -> Tuple[float, float]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int8)
    if len(scores) == 0:
        return 0.5, 0.0
    if np.all(labels == labels[0]):
        threshold = float(np.median(scores))
        pred = (scores >= threshold).astype(np.int8)
        return threshold, float(f1_score(labels, pred, average="macro", zero_division=0))

    order = np.argsort(-scores, kind="stable")
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    total_pos = int(sorted_labels.sum())
    total_neg = len(sorted_labels) - total_pos
    cum_tp = np.cumsum(sorted_labels)
    cum_fp = np.cumsum(1 - sorted_labels)

    group_ends = np.flatnonzero(
        np.r_[sorted_scores[:-1] != sorted_scores[1:], True]
    )
    tp = cum_tp[group_ends].astype(np.float64)
    fp = cum_fp[group_ends].astype(np.float64)
    fn = total_pos - tp
    tn = total_neg - fp

    f1_attack = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros_like(tp), where=(2 * tp + fp + fn) > 0)
    f1_benign = np.divide(2 * tn, 2 * tn + fp + fn, out=np.zeros_like(tn), where=(2 * tn + fp + fn) > 0)
    macro = (f1_attack + f1_benign) / 2.0
    best = int(np.argmax(macro))
    return float(sorted_scores[group_ends[best]]), float(macro[best])


def evaluate_binary_scores(
    scores: np.ndarray,
    labels: np.ndarray,
    threshold: float,
) -> Dict[str, Any]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int8)
    predictions = (scores >= float(threshold)).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()

    if len(np.unique(labels)) == 2:
        auc_roc = float(roc_auc_score(labels, scores))
        pr_auc = float(average_precision_score(labels, scores))
    else:
        auc_roc = float("nan")
        pr_auc = float("nan")

    return {
        "threshold": float(threshold),
        "f1_macro": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "auc_roc": auc_roc,
        "pr_auc": pr_auc,
        "accuracy": safe_divide(tp + tn, tp + tn + fp + fn),
        "precision": safe_divide(tp, tp + fp),
        "recall": safe_divide(tp, tp + fn),
        "specificity": safe_divide(tn, tn + fp),
        "balanced_accuracy": 0.5 * (safe_divide(tp, tp + fn) + safe_divide(tn, tn + fp)),
        "f1_attack": safe_divide(2 * tp, 2 * tp + fp + fn),
        "f1_benign": safe_divide(2 * tn, 2 * tn + fp + fn),
        "fp_rate": safe_divide(fp, fp + tn),
        "tp_rate": safe_divide(tp, tp + fn),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "scores": scores,
        "predictions": predictions,
    }


def test_curve_operating_points(
    scores: np.ndarray,
    labels: np.ndarray,
    fpr_targets: Iterable[float] = TEST_CURVE_FPR_TARGETS,
    recall_targets: Iterable[float] = TEST_CURVE_RECALL_TARGETS,
) -> List[Dict[str, Any]]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int8)
    if len(np.unique(labels)) < 2:
        return []

    fpr, tpr, roc_thresholds = roc_curve(labels, scores)
    precision, recall, pr_thresholds = precision_recall_curve(labels, scores)
    rows: List[Dict[str, Any]] = []

    for target in fpr_targets:
        eligible = np.flatnonzero(fpr <= float(target) + 1e-15)
        if len(eligible) == 0:
            chosen = 0
        else:
            chosen = eligible[np.argmax(tpr[eligible])]
        rows.append({
            "criterion": "TPR_at_test_FPR",
            "target": float(target),
            "value": float(tpr[chosen]),
            "observed_fpr": float(fpr[chosen]),
            "curve_threshold": float(roc_thresholds[chosen]),
        })

    # PR curve arrays have len(thresholds)+1. For recall targets, report the best
    # precision among test-curve points at or above the requested recall.
    for target in recall_targets:
        eligible = np.flatnonzero(recall >= float(target) - 1e-15)
        if len(eligible) == 0:
            chosen = int(np.argmax(recall))
        else:
            chosen = eligible[np.argmax(precision[eligible])]
        threshold = float("nan") if chosen >= len(pr_thresholds) else float(pr_thresholds[chosen])
        rows.append({
            "criterion": "Precision_at_test_recall",
            "target": float(target),
            "value": float(precision[chosen]),
            "observed_recall": float(recall[chosen]),
            "curve_threshold": threshold,
        })

    return rows


def per_attack_recall(
    predictions: np.ndarray,
    labels: np.ndarray,
    attack_codes: np.ndarray,
    attack_classes: np.ndarray,
) -> List[Dict[str, Any]]:
    rows = []
    for code, attack_name in enumerate(attack_classes):
        mask = (attack_codes == code) & (labels == 1)
        count = int(mask.sum())
        if count == 0:
            continue
        detected = int(predictions[mask].sum())
        rows.append({
            "attack": str(attack_name),
            "count": count,
            "detected": detected,
            "recall": float(detected / count),
        })
    return rows


In [7]:
# Cell 7 — Controlled E-GraphSAGE architecture (official core equations)

class EGraphSAGELayer(nn.Module):
    def __init__(self, node_dim_in: int, edge_dim: int, node_dim_out: int):
        super().__init__()
        # Official notebook: W_msg([source-node, edge-feature]) and
        # W_apply([self-node, mean-neighbor-message]).
        self.W_msg = nn.Linear(node_dim_in + edge_dim, node_dim_out)
        self.W_apply = nn.Linear(node_dim_in + node_dim_out, node_dim_out)

    @staticmethod
    def mean_aggregate(messages: torch.Tensor, dst: torch.Tensor, n_nodes: int) -> torch.Tensor:
        out = torch.zeros(n_nodes, messages.size(1), device=messages.device, dtype=messages.dtype)
        count = torch.zeros(n_nodes, 1, device=messages.device, dtype=messages.dtype)
        out.index_add_(0, dst, messages)
        count.index_add_(0, dst, torch.ones(len(dst), 1, device=messages.device, dtype=messages.dtype))
        return out / count.clamp_min(1.0)

    def forward(
        self,
        node_features: torch.Tensor,
        message_src: torch.Tensor,
        message_dst: torch.Tensor,
        message_edge_features: torch.Tensor,
    ) -> torch.Tensor:
        messages = self.W_msg(
            torch.cat([node_features[message_src], message_edge_features], dim=1)
        )
        aggregated = self.mean_aggregate(messages, message_dst, node_features.size(0))
        return F.relu(self.W_apply(torch.cat([node_features, aggregated], dim=1)))


class ControlledEGraphSAGE(nn.Module):
    """Two-layer edge-aware GraphSAGE + edge MLP, matching the official binary notebook.

    The official notebook first builds an undirected MultiGraph and converts it to a
    directed graph. Here, each original flow is classified once, while both directions
    are used for message passing. This preserves the intended bidirectional context
    without duplicating test predictions.
    """

    def __init__(self, edge_dim: int, hidden_dim: int = 128, dropout: float = 0.2):
        super().__init__()
        self.edge_dim = edge_dim
        self.layer1 = EGraphSAGELayer(edge_dim, edge_dim, hidden_dim)
        self.layer2 = EGraphSAGELayer(hidden_dim, edge_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.edge_classifier = nn.Linear(hidden_dim * 2, 2)

    def forward(
        self,
        edge_features: torch.Tensor,
        src: torch.Tensor,
        dst: torch.Tensor,
        n_nodes: int,
    ) -> torch.Tensor:
        node_features = torch.ones(
            n_nodes, self.edge_dim, device=edge_features.device, dtype=edge_features.dtype
        )
        message_src = torch.cat([src, dst], dim=0)
        message_dst = torch.cat([dst, src], dim=0)
        message_edge_features = torch.cat([edge_features, edge_features], dim=0)

        h = self.layer1(node_features, message_src, message_dst, message_edge_features)
        h = self.dropout(h)
        h = self.layer2(h, message_src, message_dst, message_edge_features)
        return self.edge_classifier(torch.cat([h[src], h[dst]], dim=1))


In [8]:
# Cell 8 — Controlled GraphIDS architecture (official PyG core model)

class GraphIDSSAGELayer(nn.Module):
    def __init__(
        self,
        node_dim_in: int,
        edge_dim_in: int,
        edge_dim_out: int,
        dropout_rate: float,
    ):
        super().__init__()
        # Official main-branch implementation aggregates incoming edge features,
        # transforms them to node_dim_in, concatenates source/destination node
        # embeddings and projects to an edge embedding.
        self.fc_neigh = nn.Linear(edge_dim_in, node_dim_in)
        self.fc_edge = nn.Linear(node_dim_in * 2, edge_dim_out)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        gain = nn.init.calculate_gain("relu")
        nn.init.xavier_normal_(self.fc_neigh.weight, gain=gain)
        nn.init.xavier_normal_(self.fc_edge.weight, gain=gain)
        nn.init.zeros_(self.fc_neigh.bias)
        nn.init.zeros_(self.fc_edge.bias)

    def forward(
        self,
        edge_features: torch.Tensor,
        src: torch.Tensor,
        dst: torch.Tensor,
        n_nodes: int,
    ) -> torch.Tensor:
        node_sum = torch.zeros(
            n_nodes, edge_features.size(1), device=edge_features.device, dtype=edge_features.dtype
        )
        node_count = torch.zeros(n_nodes, 1, device=edge_features.device, dtype=edge_features.dtype)
        node_sum.index_add_(0, dst, edge_features)
        node_count.index_add_(
            0, dst, torch.ones(len(dst), 1, device=edge_features.device, dtype=edge_features.dtype)
        )
        node_mean = node_sum / node_count.clamp_min(1.0)
        node_embeddings = self.relu(self.fc_neigh(node_mean))
        edge_embeddings = self.fc_edge(
            torch.cat([node_embeddings[src], node_embeddings[dst]], dim=1)
        )
        return self.dropout(edge_embeddings)


class LearnablePositionalEncoding(nn.Module):
    def __init__(self, embed_dim: int, max_len: int):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(max_len, embed_dim))
        nn.init.xavier_uniform_(self.pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[: x.size(1), :]


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, embed_dim: int, max_len: int):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float()
            * (-torch.log(torch.tensor(10000.0)) / embed_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[: x.size(1), :]


class GraphIDSTransformerAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        embed_dim: int,
        num_heads: int,
        num_layers: int,
        dropout: float,
        window_size: int,
        positional_encoding: str,
        mask_ratio: float,
        feedforward_dim: int = 256,
    ):
        super().__init__()
        if positional_encoding == "learnable":
            self.positional_encoder = LearnablePositionalEncoding(embed_dim, window_size)
        elif positional_encoding == "sinusoidal":
            self.positional_encoder = SinusoidalPositionalEncoding(embed_dim, window_size)
        else:
            self.positional_encoder = nn.Identity()

        self.input_projection = nn.Linear(input_dim, embed_dim)
        self.mask_ratio = mask_ratio
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_projection = nn.Linear(embed_dim, input_dim)
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        nn.init.xavier_uniform_(self.input_projection.weight)
        nn.init.zeros_(self.input_projection.bias)
        nn.init.xavier_uniform_(self.output_projection.weight)
        nn.init.zeros_(self.output_projection.bias)
        for module in [self.encoder, self.decoder]:
            for name, param in module.named_parameters():
                if "weight" in name and param.dim() > 1:
                    nn.init.xavier_uniform_(param)
                elif "bias" in name:
                    nn.init.zeros_(param)

    def forward(
        self,
        src: torch.Tensor,
        padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        src = self.positional_encoder(self.input_projection(src))
        src_key_padding_mask = None
        if padding_mask is not None:
            if padding_mask.dtype != torch.bool:
                padding_mask = padding_mask.bool()
            src_key_padding_mask = ~torch.any(padding_mask, dim=-1)

        if self.training and self.mask_ratio > 0:
            seq_len = src.size(1)
            mask = torch.triu(
                torch.ones(seq_len, seq_len, device=src.device, dtype=torch.bool),
                diagonal=1,
            )
            mask = mask & (
                torch.rand(seq_len, seq_len, device=src.device) < self.mask_ratio
            )
            attention_mask = mask | mask.T
        else:
            attention_mask = None

        memory = self.encoder(
            src,
            mask=attention_mask,
            src_key_padding_mask=src_key_padding_mask,
        )
        output = self.decoder(
            src,
            memory,
            memory_key_padding_mask=src_key_padding_mask,
            tgt_mask=attention_mask,
            tgt_key_padding_mask=src_key_padding_mask,
        )
        return self.output_projection(output)


class ControlledGraphIDS(nn.Module):
    def __init__(self, edge_dim: int, config: Dict[str, Any]):
        super().__init__()
        self.encoder = GraphIDSSAGELayer(
            node_dim_in=edge_dim,
            edge_dim_in=edge_dim,
            edge_dim_out=int(config["edim_out"]),
            dropout_rate=float(config["dropout"]),
        )
        self.transformer = GraphIDSTransformerAutoencoder(
            input_dim=int(config["edim_out"]),
            embed_dim=int(config["ae_embedding_dim"]),
            num_heads=int(config["num_heads"]),
            num_layers=int(config["num_layers"]),
            dropout=float(config["ae_dropout"]),
            window_size=int(config["sequence_window"]),
            positional_encoding=str(config["positional_encoding"]),
            mask_ratio=float(config["mask_ratio"]),
            feedforward_dim=int(config["ae_feedforward_dim"]),
        )


def iter_sequence_batches(
    embeddings: torch.Tensor,
    sequence_window: int,
    sequence_step: int,
    batch_size: int,
):
    sequences = []
    masks = []
    positions = []
    start = 0
    while start < len(embeddings):
        end = min(start + sequence_window, len(embeddings))
        seq = embeddings[start:end]
        mask = torch.ones_like(seq, dtype=torch.bool)
        sequences.append(seq)
        masks.append(mask)
        positions.append((start, end))
        if end == len(embeddings):
            break
        start += sequence_step

    for batch_start in range(0, len(sequences), batch_size):
        batch_sequences = sequences[batch_start:batch_start + batch_size]
        batch_masks = masks[batch_start:batch_start + batch_size]
        batch_positions = positions[batch_start:batch_start + batch_size]
        max_len = max(seq.size(0) for seq in batch_sequences)
        dim = embeddings.size(1)
        padded = torch.zeros(
            len(batch_sequences), max_len, dim,
            device=embeddings.device, dtype=embeddings.dtype,
        )
        padded_mask = torch.zeros(
            len(batch_sequences), max_len, dim,
            device=embeddings.device, dtype=torch.bool,
        )
        for i, (seq, mask) in enumerate(zip(batch_sequences, batch_masks)):
            padded[i, : seq.size(0)] = seq
            padded_mask[i, : seq.size(0)] = mask
        yield padded, padded_mask, batch_positions


def graphids_reconstruction_errors(
    outputs: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    squared = ((outputs - targets) ** 2) * mask
    valid_counts = mask.sum(dim=-1)
    valid_tokens = valid_counts > 0
    mean_errors = torch.zeros_like(valid_counts, dtype=torch.float32)
    mean_errors[valid_tokens] = (
        squared.sum(dim=-1)[valid_tokens]
        / valid_counts[valid_tokens].clamp_min(1)
    )
    return torch.nan_to_num(mean_errors, nan=0.0, posinf=1e6, neginf=-1e6)


In [9]:
# Cell 9 — Public-baseline training and inference helpers

def egraph_node_tensors(window: PublicGraphWindow, device: torch.device):
    return (
        window.src_endpoint.to(device, non_blocking=True),
        window.dst_endpoint.to(device, non_blocking=True),
        window.n_endpoint_nodes,
    )


def graphids_node_tensors(window: PublicGraphWindow, device: torch.device):
    return (
        window.src_ip.to(device, non_blocking=True),
        window.dst_ip.to(device, non_blocking=True),
        window.n_ip_nodes,
    )


def train_egraph_epoch(
    model: ControlledEGraphSAGE,
    windows: List[PublicGraphWindow],
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
) -> float:
    model.train()
    total_loss = 0.0
    total_rows = 0
    order = np.random.permutation(len(windows))
    for window_index in order:
        window = windows[int(window_index)]
        edge_feat = window.edge_feat.to(DEVICE, non_blocking=True)
        labels = window.labels.to(DEVICE, non_blocking=True)
        src, dst, n_nodes = egraph_node_tensors(window, DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(edge_feat, src, dst, n_nodes)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        total_loss += float(loss.item()) * len(labels)
        total_rows += len(labels)
    return total_loss / max(total_rows, 1)


@torch.inference_mode()
def collect_egraph_scores(
    model: ControlledEGraphSAGE,
    windows: List[PublicGraphWindow],
) -> Dict[str, np.ndarray]:
    model.eval()
    scores, labels, attacks = [], [], []
    for window in windows:
        edge_feat = window.edge_feat.to(DEVICE, non_blocking=True)
        src, dst, n_nodes = egraph_node_tensors(window, DEVICE)
        logits = model(edge_feat, src, dst, n_nodes)
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        scores.append(probabilities.cpu().numpy())
        labels.append(window.labels.numpy())
        attacks.append(window.attack_codes.numpy())
    return {
        "scores": np.concatenate(scores).astype(np.float64),
        "labels": np.concatenate(labels).astype(np.int8),
        "attack_codes": np.concatenate(attacks).astype(np.int16),
    }


def train_graphids_epoch(
    model: ControlledGraphIDS,
    windows: List[PublicGraphWindow],
    optimizer: torch.optim.Optimizer,
    config: Dict[str, Any],
) -> float:
    model.train()
    total_loss = 0.0
    total_windows = 0
    order = np.random.permutation(len(windows))

    for window_index in order:
        window = windows[int(window_index)]
        edge_feat = window.edge_feat.to(DEVICE, non_blocking=True)
        src, dst, n_nodes = graphids_node_tensors(window, DEVICE)
        embeddings = model.encoder(edge_feat, src, dst, n_nodes)

        optimizer.zero_grad(set_to_none=True)
        sequence_losses = []
        for seq_batch, mask_batch, _ in iter_sequence_batches(
            embeddings,
            sequence_window=int(config["sequence_window"]),
            sequence_step=int(config["sequence_step"]),
            batch_size=int(config["ae_batch_size"]),
        ):
            outputs = model.transformer(seq_batch, mask_batch)
            loss_elements = ((outputs - seq_batch) ** 2) * mask_batch
            loss = loss_elements.sum() / mask_batch.sum().clamp_min(1)
            sequence_losses.append(loss)

        if not sequence_losses:
            continue
        # Official code intentionally keeps gradients through the target embedding.
        window_loss = torch.stack(sequence_losses).mean()
        window_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        total_loss += float(window_loss.item())
        total_windows += 1

    return total_loss / max(total_windows, 1)


@torch.inference_mode()
def collect_graphids_scores(
    model: ControlledGraphIDS,
    windows: List[PublicGraphWindow],
    config: Dict[str, Any],
) -> Dict[str, np.ndarray]:
    model.eval()
    all_errors, all_labels, all_attacks = [], [], []

    for window in windows:
        edge_feat = window.edge_feat.to(DEVICE, non_blocking=True)
        src, dst, n_nodes = graphids_node_tensors(window, DEVICE)
        embeddings = model.encoder(edge_feat, src, dst, n_nodes)

        window_errors = torch.empty(len(embeddings), device=DEVICE, dtype=torch.float32)
        filled = torch.zeros(len(embeddings), device=DEVICE, dtype=torch.bool)

        for seq_batch, mask_batch, positions in iter_sequence_batches(
            embeddings,
            sequence_window=int(config["sequence_window"]),
            sequence_step=int(config["sequence_step"]),
            batch_size=int(config["ae_batch_size"]),
        ):
            outputs = model.transformer(seq_batch, mask_batch)
            errors = graphids_reconstruction_errors(outputs, seq_batch, mask_batch)
            for i, (start, end) in enumerate(positions):
                length = end - start
                # Step equals window in the official v2 configs; no overlap is expected.
                window_errors[start:end] = errors[i, :length]
                filled[start:end] = True

        if not bool(filled.all()):
            raise RuntimeError("GraphIDS sequence reconstruction did not cover every flow.")

        all_errors.append(window_errors.cpu().numpy())
        all_labels.append(window.labels.numpy())
        all_attacks.append(window.attack_codes.numpy())

    return {
        "scores": np.concatenate(all_errors).astype(np.float64),
        "labels": np.concatenate(all_labels).astype(np.int8),
        "attack_codes": np.concatenate(all_attacks).astype(np.int16),
    }


In [10]:
# Cell 10 — Robust task runner with resume/checkpoints

def create_public_model_and_optimizer(
    baseline_name: str,
    dataset_name: str,
    edge_dim: int,
    positive_class_weight: float,
):
    if baseline_name == "egraphsage_public":
        config = dict(EGRAPH_CONFIG)
        model = ControlledEGraphSAGE(
            edge_dim=edge_dim,
            hidden_dim=int(config["hidden_dim"]),
            dropout=float(config["dropout"]),
        ).to(DEVICE)
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=float(config["learning_rate"]),
            weight_decay=float(config["weight_decay"]),
        )
        class_weights = torch.tensor(
            [1.0, float(positive_class_weight)],
            device=DEVICE,
            dtype=torch.float32,
        )
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        return model, optimizer, criterion, config

    if baseline_name == "graphids_public":
        config = dict(GRAPHIDS_CONFIG[dataset_name])
        model = ControlledGraphIDS(edge_dim=edge_dim, config=config).to(DEVICE)
        optimizer = torch.optim.AdamW(
            [
                {
                    "params": model.encoder.parameters(),
                    "weight_decay": float(config["encoder_weight_decay"]),
                },
                {
                    "params": model.transformer.parameters(),
                    "weight_decay": float(config["ae_weight_decay"]),
                },
            ],
            lr=float(config["learning_rate"]),
        )
        return model, optimizer, None, config

    raise KeyError(baseline_name)


def run_public_baseline_task(
    bundle: FixedSubsetBundle,
    baseline_name: str,
    seed: int,
    train_windows: List[PublicGraphWindow],
    graphids_train_windows: List[PublicGraphWindow],
    val_windows: List[PublicGraphWindow],
    test_windows: List[PublicGraphWindow],
    positive_class_weight: float,
    split_fingerprint: str,
) -> Dict[str, Any]:
    model_dir = OUTPUT_ROOT / bundle.name / PROTOCOL_NAME / baseline_name / f"seed_{seed}"
    model_dir.mkdir(parents=True, exist_ok=True)

    config_for_fingerprint = (
        dict(EGRAPH_CONFIG)
        if baseline_name == "egraphsage_public"
        else dict(GRAPHIDS_CONFIG[bundle.name])
    )
    expected_fingerprint = task_fingerprint(
        dataset_name=bundle.name,
        baseline_name=baseline_name,
        seed=seed,
        feature_names=bundle.feature_names,
        subset_fingerprint=bundle.subset_fingerprint,
        split_fingerprint=split_fingerprint,
        config=config_for_fingerprint,
    )

    config_path = model_dir / "config.json"
    history_path = model_dir / "history.csv"
    best_path = model_dir / "best.pt"
    last_path = model_dir / "last.pt"
    result_path = model_dir / "test_metrics.json"
    prediction_path = model_dir / "test_predictions.npz"
    completed_path = model_dir / "completed.json"

    if completed_path.exists() and result_path.exists() and prediction_path.exists():
        completed = load_json(completed_path)
        if completed.get("config_fingerprint") == expected_fingerprint:
            print(f"  SKIP completed: {bundle.name} | {baseline_name} | seed={seed}")
            return load_json(result_path)
        raise RuntimeError(
            f"Existing task fingerprint mismatch: {model_dir}. "
            "Use a new STEP6_EXPERIMENT_TAG or remove only this task folder."
        )

    seed_everything(seed)
    model, optimizer, criterion, config = create_public_model_and_optimizer(
        baseline_name,
        bundle.name,
        len(bundle.feature_names),
        positive_class_weight,
    )
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    max_epochs = int(config["max_epochs"])
    patience = int(config["patience"])

    task_config = {
        "dataset": bundle.name,
        "protocol": PROTOCOL_NAME,
        "baseline": baseline_name,
        "seed": seed,
        "parameters": parameter_count,
        "feature_names": bundle.feature_names,
        "subset_fingerprint": bundle.subset_fingerprint,
        "split_fingerprint": split_fingerprint,
        "graph_window_size": PUBLIC_GRAPH_WINDOW_SIZE,
        "public_source": PUBLIC_SOURCE_METADATA[baseline_name],
        "controlled_config": config,
        "config_fingerprint": expected_fingerprint,
    }
    atomic_json_dump(task_config, config_path)

    start_epoch = 1
    best_selection_value = -float("inf")
    best_epoch = -1
    best_threshold = 0.5
    patience_counter = 0
    history = load_history(history_path)

    if last_path.exists():
        checkpoint = load_torch_checkpoint(last_path, DEVICE)
        if checkpoint.get("config_fingerprint") != expected_fingerprint:
            raise RuntimeError(f"Checkpoint config mismatch: {last_path}")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_selection_value = float(checkpoint["best_selection_value"])
        best_epoch = int(checkpoint["best_epoch"])
        best_threshold = float(checkpoint["best_threshold"])
        patience_counter = int(checkpoint["patience_counter"])
        restore_rng_state(checkpoint.get("rng_state"))
        print(f"  RESUME {baseline_name} seed={seed} from epoch {start_epoch}")
    else:
        print(f"  START {baseline_name} seed={seed} | params={parameter_count:,}")

    task_start = time.time()
    for epoch in range(start_epoch, max_epochs + 1) if patience_counter < patience else []:
        epoch_start = time.time()

        if baseline_name == "egraphsage_public":
            train_loss = train_egraph_epoch(model, train_windows, optimizer, criterion)
            val_collected = collect_egraph_scores(model, val_windows)
        else:
            train_loss = train_graphids_epoch(model, graphids_train_windows, optimizer, config)
            val_collected = collect_graphids_scores(model, val_windows, config)

        val_threshold, val_f1 = find_best_threshold_macro_f1_exact(
            val_collected["scores"], val_collected["labels"]
        )
        val_eval = evaluate_binary_scores(
            val_collected["scores"], val_collected["labels"], val_threshold
        )
        selection_value = (
            val_eval["f1_macro"]
            if baseline_name == "egraphsage_public"
            else val_eval["pr_auc"]
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_f1_macro": val_eval["f1_macro"],
            "val_auc_roc": val_eval["auc_roc"],
            "val_pr_auc": val_eval["pr_auc"],
            "val_threshold": val_threshold,
            "selection_value": selection_value,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.time() - epoch_start,
        }
        history = [h for h in history if int(h["epoch"]) != epoch]
        history.append(row)
        history.sort(key=lambda x: int(x["epoch"]))
        save_history(history, history_path)

        improved = selection_value > best_selection_value
        if improved:
            best_selection_value = float(selection_value)
            best_epoch = epoch
            best_threshold = float(val_threshold)
            patience_counter = 0
            atomic_torch_save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch,
                    "best_selection_value": best_selection_value,
                    "best_threshold": best_threshold,
                    "config_fingerprint": expected_fingerprint,
                },
                best_path,
            )
        else:
            patience_counter += 1

        if epoch % SAVE_LAST_EVERY == 0 or improved or epoch == max_epochs:
            atomic_torch_save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "best_selection_value": best_selection_value,
                    "best_epoch": best_epoch,
                    "best_threshold": best_threshold,
                    "patience_counter": patience_counter,
                    "rng_state": capture_rng_state(),
                    "config_fingerprint": expected_fingerprint,
                },
                last_path,
            )

        print(
            f"    Ep {epoch:03d}/{max_epochs} | loss={train_loss:.6f} | "
            f"valF1={val_eval['f1_macro']:.4f} | AUC={val_eval['auc_roc']:.4f} | "
            f"PR={val_eval['pr_auc']:.4f} | thr={val_threshold:.6f} | "
            f"pat={patience_counter}/{patience} | {time.time() - epoch_start:.1f}s"
        )
        if patience_counter >= patience:
            print(
                f"    Early stop. Best epoch={best_epoch}, "
                f"selection={best_selection_value:.6f}"
            )
            break

    if not best_path.exists():
        raise RuntimeError(f"Best checkpoint was not created: {best_path}")

    best = load_torch_checkpoint(best_path, DEVICE)
    model.load_state_dict(best["model_state_dict"])

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    prediction_start = time.perf_counter()
    if baseline_name == "egraphsage_public":
        test_collected = collect_egraph_scores(model, test_windows)
    else:
        test_collected = collect_graphids_scores(model, test_windows, config)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    prediction_seconds = time.perf_counter() - prediction_start

    test_eval = evaluate_binary_scores(
        test_collected["scores"], test_collected["labels"], best_threshold
    )
    attack_rows = per_attack_recall(
        test_eval["predictions"],
        test_collected["labels"],
        test_collected["attack_codes"],
        bundle.attack_classes,
    )

    result = {
        "dataset": bundle.name,
        "protocol": PROTOCOL_NAME,
        "model": baseline_name,
        "seed": seed,
        "parameters": parameter_count,
        "best_epoch": best_epoch,
        "selection_metric": config["selection_metric"],
        "validation": {
            "selection_value": best_selection_value,
            "threshold": best_threshold,
        },
        "test": {
            key: value for key, value in test_eval.items()
            if key not in {"scores", "predictions"}
        },
        "per_attack": attack_rows,
        "prediction_seconds": prediction_seconds,
        "elapsed_seconds_history": float(
            sum(float(row.get("epoch_seconds", 0.0)) for row in history)
        ),
        "epochs_recorded": int(len(history)),
        "subset_fingerprint": bundle.subset_fingerprint,
        "split_fingerprint": split_fingerprint,
        "public_source": PUBLIC_SOURCE_METADATA[baseline_name],
        "config_fingerprint": expected_fingerprint,
    }
    atomic_json_dump(result, result_path)

    np.savez_compressed(
        prediction_path,
        scores=test_eval["scores"].astype(np.float32),
        predictions=test_eval["predictions"].astype(np.int8),
        labels=test_collected["labels"].astype(np.int8),
        attack_codes=test_collected["attack_codes"].astype(np.int16),
        attack_classes=bundle.attack_classes.astype(str),
        threshold=np.asarray([best_threshold], dtype=np.float64),
    )

    atomic_json_dump(
        {
            "completed": True,
            "dataset": bundle.name,
            "baseline": baseline_name,
            "seed": seed,
            "completed_at_unix": time.time(),
            "config_fingerprint": expected_fingerprint,
        },
        completed_path,
    )

    print(
        f"  DONE {baseline_name} seed={seed} | "
        f"F1={result['test']['f1_macro']:.4f} | "
        f"AUC={result['test']['auc_roc']:.4f} | "
        f"PR={result['test']['pr_auc']:.4f} | "
        f"FPR={100 * result['test']['fp_rate']:.4f}%"
    )

    del model, optimizer, test_collected, test_eval
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


In [11]:
# Cell 11 — Run Step 6 public baselines

def run_dataset_step6(dataset_name: str) -> None:
    print("\n" + "=" * 110)
    print(f"STEP 6 DATASET: {dataset_name}")
    print("=" * 110)

    bundle = load_fixed_subset_bundle(dataset_name)
    train_idx, val_idx, test_idx, scaler, split_meta = load_frozen_step5_artifacts(bundle)
    split_fingerprint = str(split_meta["split_fingerprint"])

    y_train = bundle.labels[train_idx]
    n_positive = int((y_train == 1).sum())
    n_negative = int((y_train == 0).sum())
    positive_class_weight = n_negative / max(n_positive, 1)
    print(f"E-GraphSAGE class weight (attack): {positive_class_weight:.6f}")

    dataset_output = OUTPUT_ROOT / dataset_name
    dataset_output.mkdir(parents=True, exist_ok=True)
    atomic_json_dump(
        {
            "dataset": dataset_name,
            "step5_confirmation_root": str(step5_dataset_dir(dataset_name)),
            "subset_fingerprint": bundle.subset_fingerprint,
            "split_fingerprint": split_fingerprint,
            "fixed_split_seed": BENCHMARK_SPLIT_SEED,
            "model_seeds": MODEL_SEEDS,
            "public_baselines": PUBLIC_BASELINES_TO_RUN,
            "feature_names": bundle.feature_names,
            "public_graph_window_size": PUBLIC_GRAPH_WINDOW_SIZE,
            "comparison_scope": "controlled public-baseline reimplementation",
            "public_sources": PUBLIC_SOURCE_METADATA,
        },
        dataset_output / "experiment_manifest.json",
    )

    train_windows = prepare_public_graph_windows(bundle, train_idx, scaler, "Train mixed")
    benign_train_idx = select_benign_train_indices(bundle, train_idx)
    graphids_train_windows = prepare_public_graph_windows(
        bundle, benign_train_idx, scaler, "Train benign-only (GraphIDS)"
    )
    val_windows = prepare_public_graph_windows(bundle, val_idx, scaler, "Validation")
    test_windows = prepare_public_graph_windows(bundle, test_idx, scaler, "Test")

    for seed in MODEL_SEEDS:
        print(f"\n--- MODEL SEED {seed} | frozen split {BENCHMARK_SPLIT_SEED} ---")
        for baseline_name in PUBLIC_BASELINES_TO_RUN:
            run_public_baseline_task(
                bundle=bundle,
                baseline_name=baseline_name,
                seed=seed,
                train_windows=train_windows,
                graphids_train_windows=graphids_train_windows,
                val_windows=val_windows,
                test_windows=test_windows,
                positive_class_weight=positive_class_weight,
                split_fingerprint=split_fingerprint,
            )

    del train_windows, graphids_train_windows, val_windows, test_windows, scaler, bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


for dataset_name in DATASETS_TO_RUN:
    run_dataset_step6(dataset_name)

print("\nAll requested Step 6 public-baseline tasks completed or resumed from checkpoints.")



STEP 6 DATASET: CIC_V2

Loading fixed Step-5 subset for CIC_V2
CSV: /content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv
Subset rows: 496,384 | features: 39
  CSV chunks=10 | selected=131,692/496,384
  CSV chunks=20 | selected=263,031/496,384
  CSV chunks=30 | selected=394,663/496,384
  CSV chunks=38 | selected=496,384/496,384
Loaded fixed subset in 1.7 min
Label counts: {np.int8(0): np.int64(349993), np.int8(1): np.int64(146391)}
Frozen split: train=361,202, val=40,519, test=94,663 | fingerprint=58cfcf6084a427575c09
E-GraphSAGE class weight (attack): 2.225133
  Train mixed: 73/73 windows
Prepared Train mixed: 73 windows in 0.0 min
GraphIDS benign-only train rows: 249,206/361,202
  Train benign-only (GraphIDS): 50/50 windows
Prepared Train benign-only (GraphIDS): 50 windows in 0.0 min
  Validation: 9/9 windows
Prepared Validation: 9 windows in 0.0 min
  Test: 19/19 windows
Prepared Test: 19 windows in 0.0 min

--- MODEL SEED 42 | frozen split 2027 ---
  START egraphsage_public seed=42 | p

In [12]:
# Cell 12 — Load Step-5 confirmation and Step-6 public results

STEP5_MODELS = ["mlp", "no_graph", "full", "full_shuffled_edges"]
PUBLIC_MODELS = ["egraphsage_public", "graphids_public"]
ALL_COMPARISON_MODELS = STEP5_MODELS + PUBLIC_MODELS


def normalize_result_row(result: Dict[str, Any], source_stage: str) -> Dict[str, Any]:
    test = result["test"]
    model_name = result.get("model", result.get("variant"))
    return {
        "dataset": result["dataset"],
        "protocol": result.get("protocol", PROTOCOL_NAME),
        "model": model_name,
        "seed": int(result["seed"]),
        "parameters": float(result.get("parameters", np.nan)),
        "best_epoch": int(result.get("best_epoch", -1)),
        "f1_macro": float(test["f1_macro"]),
        "auc_roc": float(test["auc_roc"]),
        "pr_auc": float(test["pr_auc"]),
        "accuracy": float(test.get("accuracy", np.nan)),
        "precision": float(test.get("precision", np.nan)),
        "recall": float(test.get("recall", test.get("tp_rate", np.nan))),
        "specificity": float(test.get("specificity", np.nan)),
        "balanced_accuracy": float(test.get("balanced_accuracy", np.nan)),
        "f1_attack": float(test.get("f1_attack", np.nan)),
        "f1_benign": float(test.get("f1_benign", np.nan)),
        "fp_rate": float(test["fp_rate"]),
        "tp_rate": float(test["tp_rate"]),
        "tn": int(test["tn"]),
        "fp": int(test["fp"]),
        "fn": int(test["fn"]),
        "tp": int(test["tp"]),
        "source_stage": source_stage,
    }


def result_and_prediction_paths(dataset: str, model: str, seed: int) -> Tuple[Path, Path]:
    if model in STEP5_MODELS:
        root = (
            STEP5_CONFIRMATION_ROOT / dataset / PROTOCOL_NAME / model / f"seed_{seed}"
        )
    else:
        root = OUTPUT_ROOT / dataset / PROTOCOL_NAME / model / f"seed_{seed}"
    return root / "test_metrics.json", root / "test_predictions.npz"


def load_all_comparison_outputs():
    result_rows = []
    attack_rows = []
    completeness_rows = []

    for dataset in DATA_PATH_CANDIDATES:
        for model in ALL_COMPARISON_MODELS:
            for seed in MODEL_SEEDS:
                result_path, prediction_path = result_and_prediction_paths(dataset, model, seed)
                complete = result_path.exists() and prediction_path.exists()
                completeness_rows.append({
                    "dataset": dataset,
                    "model": model,
                    "seed": seed,
                    "complete": complete,
                    "result_path": str(result_path),
                    "prediction_path": str(prediction_path),
                })
                if not complete:
                    continue
                result = load_json(result_path)
                stage = "step5_confirmation" if model in STEP5_MODELS else "step6_public"
                result_rows.append(normalize_result_row(result, stage))
                for attack in result.get("per_attack", []):
                    attack_rows.append({
                        "dataset": dataset,
                        "model": model,
                        "seed": seed,
                        "attack": attack["attack"],
                        "support": int(attack["count"]),
                        "recall": float(attack["recall"]),
                        "source_stage": stage,
                    })

    return (
        pd.DataFrame(result_rows),
        pd.DataFrame(attack_rows),
        pd.DataFrame(completeness_rows),
    )


all_runs_df, per_attack_df, completeness_df = load_all_comparison_outputs()
print("Loaded run rows:", len(all_runs_df))
display(completeness_df)


Loaded run rows: 36


,dataset,model,seed,complete,result_path,prediction_path
0,UNSW_V2,mlp,42,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
1,UNSW_V2,mlp,123,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
2,UNSW_V2,mlp,456,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
3,UNSW_V2,no_graph,42,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
4,UNSW_V2,no_graph,123,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
5,UNSW_V2,no_graph,456,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
6,UNSW_V2,full,42,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
7,UNSW_V2,full,123,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
8,UNSW_V2,full,456,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
9,UNSW_V2,full_shuffled_edges,42,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...


In [13]:
# Cell 13 — Final aggregation, paired comparisons, and curve summaries

def sample_std(series: pd.Series) -> float:
    return float(series.std(ddof=1)) if len(series) > 1 else 0.0


def mean_std_text(mean: float, std: float) -> str:
    return f"{mean:.6f} ± {std:.6f}"


if all_runs_df.empty:
    raise RuntimeError("No Step-5/Step-6 results were found.")

expected_public = len(DATA_PATH_CANDIDATES) * len(PUBLIC_MODELS) * len(MODEL_SEEDS)
public_complete = completeness_df[completeness_df["model"].isin(PUBLIC_MODELS)]["complete"].sum()
print(f"Public tasks complete: {public_complete}/{expected_public}")
if STRICT_COMPLETENESS and public_complete != expected_public:
    raise RuntimeError("Step 6 public-baseline tasks are incomplete.")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
all_runs_df.sort_values(["dataset", "model", "seed"]).to_csv(
    OUTPUT_ROOT / "step6_all_runs.csv", index=False
)
completeness_df.to_csv(OUTPUT_ROOT / "step6_run_completeness.csv", index=False)

metrics = ["f1_macro", "auc_roc", "pr_auc", "fp_rate", "tp_rate"]
summary_rows = []
for (dataset, model), group in all_runs_df.groupby(["dataset", "model"], sort=True):
    row = {
        "dataset": dataset,
        "model": model,
        "n_seeds": len(group),
        "parameters": float(group["parameters"].mean()),
    }
    for metric in metrics:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_std"] = sample_std(group[metric])
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_ROOT / "step6_model_summary.csv", index=False)

paper_rows = []
for _, row in summary_df.iterrows():
    paper_rows.append({
        "dataset": row["dataset"],
        "model": row["model"],
        "n_seeds": int(row["n_seeds"]),
        "parameters": row["parameters"],
        "f1_macro": mean_std_text(row["f1_macro_mean"], row["f1_macro_std"]),
        "auc_roc": mean_std_text(row["auc_roc_mean"], row["auc_roc_std"]),
        "pr_auc": mean_std_text(row["pr_auc_mean"], row["pr_auc_std"]),
        "fp_rate": mean_std_text(row["fp_rate_mean"], row["fp_rate_std"]),
        "tp_rate": mean_std_text(row["tp_rate_mean"], row["tp_rate_std"]),
    })
paper_df = pd.DataFrame(paper_rows)
paper_df.to_csv(OUTPUT_ROOT / "step6_paper_main_results_table.csv", index=False)
with open(OUTPUT_ROOT / "step6_paper_main_results_table.tex", "w", encoding="utf-8") as f:
    f.write(paper_df.to_latex(index=False, escape=True))

# Paired comparisons using identical model seeds.
paired_rows = []
for dataset in sorted(all_runs_df["dataset"].unique()):
    full = all_runs_df[(all_runs_df.dataset == dataset) & (all_runs_df.model == "full")]
    for baseline in ["egraphsage_public", "graphids_public"]:
        other = all_runs_df[(all_runs_df.dataset == dataset) & (all_runs_df.model == baseline)]
        merged = full.merge(other, on=["dataset", "seed"], suffixes=("_full", "_baseline"))
        for _, run in merged.iterrows():
            paired_rows.append({
                "dataset": dataset,
                "comparison": f"full_minus_{baseline}",
                "seed": int(run["seed"]),
                "delta_f1_macro": run["f1_macro_full"] - run["f1_macro_baseline"],
                "delta_auc_roc": run["auc_roc_full"] - run["auc_roc_baseline"],
                "delta_pr_auc": run["pr_auc_full"] - run["pr_auc_baseline"],
                "delta_fp_rate": run["fp_rate_full"] - run["fp_rate_baseline"],
                "delta_tp_rate": run["tp_rate_full"] - run["tp_rate_baseline"],
                "delta_fp": run["fp_full"] - run["fp_baseline"],
                "delta_fn": run["fn_full"] - run["fn_baseline"],
            })
paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv(OUTPUT_ROOT / "step6_paired_differences.csv", index=False)

paired_summary_rows = []
if not paired_df.empty:
    for (dataset, comparison), group in paired_df.groupby(["dataset", "comparison"]):
        paired_summary_rows.append({
            "dataset": dataset,
            "comparison": comparison,
            "n_seeds": len(group),
            "delta_f1_macro_mean": group.delta_f1_macro.mean(),
            "delta_f1_macro_std": sample_std(group.delta_f1_macro),
            "f1_wins": int((group.delta_f1_macro > 0).sum()),
            "delta_auc_roc_mean": group.delta_auc_roc.mean(),
            "delta_pr_auc_mean": group.delta_pr_auc.mean(),
            "delta_pr_auc_std": sample_std(group.delta_pr_auc),
            "pr_auc_wins": int((group.delta_pr_auc > 0).sum()),
            "delta_fp_rate_mean": group.delta_fp_rate.mean(),
            "lower_fpr_wins": int((group.delta_fp_rate < 0).sum()),
            "delta_tp_rate_mean": group.delta_tp_rate.mean(),
            "higher_tpr_wins": int((group.delta_tp_rate > 0).sum()),
            "delta_fp_mean": group.delta_fp.mean(),
            "delta_fn_mean": group.delta_fn.mean(),
        })
paired_summary_df = pd.DataFrame(paired_summary_rows)
paired_summary_df.to_csv(OUTPUT_ROOT / "step6_paired_summary.csv", index=False)

# Per-attack aggregation.
if not per_attack_df.empty:
    attack_summary = (
        per_attack_df.groupby(["dataset", "model", "attack"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            support_mean=("support", "mean"),
            recall_mean=("recall", "mean"),
            recall_std=("recall", lambda x: sample_std(pd.Series(x))),
            recall_min=("recall", "min"),
            recall_max=("recall", "max"),
        )
    )
    per_attack_df.to_csv(OUTPUT_ROOT / "step6_per_attack_all_runs.csv", index=False)
    attack_summary.to_csv(OUTPUT_ROOT / "step6_per_attack_summary.csv", index=False)

# Test-curve operating summaries from saved raw scores for every model.
curve_rows = []
for _, run in all_runs_df.iterrows():
    _, prediction_path = result_and_prediction_paths(run.dataset, run.model, int(run.seed))
    data = np.load(prediction_path, allow_pickle=True)
    scores = np.asarray(data["scores"], dtype=np.float64)
    labels = np.asarray(data["labels"], dtype=np.int8)
    for point in test_curve_operating_points(scores, labels):
        curve_rows.append({
            "dataset": run.dataset,
            "model": run.model,
            "seed": int(run.seed),
            **point,
        })
curve_df = pd.DataFrame(curve_rows)
curve_df.to_csv(OUTPUT_ROOT / "step6_test_curve_operating_points.csv", index=False)
if not curve_df.empty:
    curve_summary = (
        curve_df.groupby(["dataset", "model", "criterion", "target"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            value_mean=("value", "mean"),
            value_std=("value", lambda x: sample_std(pd.Series(x))),
        )
    )
    curve_summary.to_csv(OUTPUT_ROOT / "step6_test_curve_operating_summary.csv", index=False)

print("\nPaper-ready main table")
display(paper_df.sort_values(["dataset", "model"]))
print("\nPaired Full-vs-public summary")
display(paired_summary_df if paired_summary_rows else pd.DataFrame())


Public tasks complete: 12/12

Paper-ready main table


,dataset,model,n_seeds,parameters,f1_macro,auc_roc,pr_auc,fp_rate,tp_rate
0,CIC_V2,egraphsage_public,3,86530.0,0.971101 ± 0.000177,0.971678 ± 0.000138,0.965095 ± 0.000220,0.003826 ± 0.000157,0.926171 ± 0.000956
1,CIC_V2,full,3,157196.0,0.973577 ± 0.001140,0.990513 ± 0.001349,0.983966 ± 0.001665,0.000223 ± 0.000056,0.922586 ± 0.003318
2,CIC_V2,full_shuffled_edges,3,157196.0,0.974191 ± 0.000989,0.989301 ± 0.000192,0.982491 ± 0.000237,0.000533 ± 0.000155,0.925279 ± 0.003312
3,CIC_V2,graphids_public,3,57144.0,0.840514 ± 0.095141,0.907778 ± 0.059620,0.737597 ± 0.203577,0.072384 ± 0.060086,0.747168 ± 0.158960
4,CIC_V2,mlp,3,70945.0,0.973460 ± 0.005015,0.980547 ± 0.000284,0.970819 ± 0.000427,0.003830 ± 0.005755,0.932932 ± 0.002577
5,CIC_V2,no_graph,3,97096.0,0.975413 ± 0.000957,0.980640 ± 0.000167,0.970755 ± 0.000398,0.001396 ± 0.001015,0.931381 ± 0.000806
6,UNSW_V2,egraphsage_public,3,86530.0,0.995998 ± 0.000050,0.998922 ± 0.000120,0.996606 ± 0.000254,0.004820 ± 0.000122,0.999685 ± 0.000141
7,UNSW_V2,full,3,157196.0,0.994810 ± 0.001947,0.999194 ± 0.000054,0.997421 ± 0.000213,0.006057 ± 0.002207,0.999145 ± 0.000656
8,UNSW_V2,full_shuffled_edges,3,157196.0,0.992907 ± 0.000230,0.998984 ± 0.000036,0.996844 ± 0.000145,0.008804 ± 0.000302,0.999977 ± 0.000039
9,UNSW_V2,graphids_public,3,92744.0,0.786651 ± 0.124556,0.888632 ± 0.064347,0.671610 ± 0.171080,0.152303 ± 0.042718,0.752891 ± 0.266582



Paired Full-vs-public summary


,dataset,comparison,n_seeds,delta_f1_macro_mean,delta_f1_macro_std,f1_wins,delta_auc_roc_mean,delta_pr_auc_mean,delta_pr_auc_std,pr_auc_wins,delta_fp_rate_mean,lower_fpr_wins,delta_tp_rate_mean,higher_tpr_wins,delta_fp_mean,delta_fn_mean
0,CIC_V2,full_minus_egraphsage_public,3,0.002476,0.001125,3,0.018835,0.018871,0.001790,3,-0.003603,3,-0.003585,0,-259.000000,81.666667
1,CIC_V2,full_minus_graphids_public,3,0.133064,0.095367,3,0.082736,0.246369,0.205215,3,-0.072162,3,0.175418,3,-5187.333333,-3995.666667
2,UNSW_V2,full_minus_egraphsage_public,3,-0.001189,0.001911,0,0.000272,0.000815,0.000136,3,0.001238,1,-0.000540,1,41.000000,8.000000
3,UNSW_V2,full_minus_graphids_public,3,0.208159,0.126039,3,0.110562,0.325811,0.171292,3,-0.146245,3,0.246254,3,-4844.666667,-3648.000000


In [14]:
# Cell 14 — Final decision file, Markdown summary, and result bundle

def model_summary_record(dataset: str, model: str) -> Optional[pd.Series]:
    rows = summary_df[(summary_df.dataset == dataset) & (summary_df.model == model)]
    return None if rows.empty else rows.iloc[0]


decision_rows = []
for dataset in sorted(all_runs_df.dataset.unique()):
    full = model_summary_record(dataset, "full")
    if full is None:
        continue
    public_available = [
        model for model in PUBLIC_MODELS
        if model_summary_record(dataset, model) is not None
    ]
    complete_public = len(public_available) == len(PUBLIC_MODELS)

    messages = []
    for model in public_available:
        baseline = model_summary_record(dataset, model)
        messages.append(
            f"Full vs {model}: ΔF1={full.f1_macro_mean - baseline.f1_macro_mean:+.6f}, "
            f"ΔPR={full.pr_auc_mean - baseline.pr_auc_mean:+.6f}, "
            f"ΔFPR={full.fp_rate_mean - baseline.fp_rate_mean:+.6f}."
        )

    decision_rows.append({
        "dataset": dataset,
        "public_tasks_complete": complete_public,
        "public_models_available": ",".join(public_available),
        "decision": "PUBLIC_BASELINE_COMPARISON_COMPLETE" if complete_public else "INCOMPLETE_PUBLIC_BASELINES",
        "paper_message": " ".join(messages),
    })

decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(OUTPUT_ROOT / "step6_final_decision.csv", index=False)

summary_lines = [
    "# THAT-IDS Step 6 controlled public-baseline summary",
    "",
    f"- Step-5 benchmark: `{STEP5_EXPERIMENT_TAG}/{STEP5_PHASE}`",
    f"- Step-6 tag: `{STEP6_EXPERIMENT_TAG}`",
    f"- Fixed split seed: `{BENCHMARK_SPLIT_SEED}`",
    f"- Model seeds: `{MODEL_SEEDS}`",
    f"- Public baselines: `{PUBLIC_MODELS}`",
    "- Comparison type: controlled reimplementation on frozen Step-5 rows/split/scaler",
    "",
]
for _, row in decision_df.iterrows():
    summary_lines.extend([
        f"## {row['dataset']}",
        f"- Decision: **{row['decision']}**",
        f"- {row['paper_message']}",
        "",
    ])

summary_path = OUTPUT_ROOT / "step6_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

bundle_path = OUTPUT_ROOT / "THAT_IDS_step6_public_baseline_results_bundle.zip"
small_files = [
    "step6_all_runs.csv",
    "step6_run_completeness.csv",
    "step6_model_summary.csv",
    "step6_paper_main_results_table.csv",
    "step6_paper_main_results_table.tex",
    "step6_paired_differences.csv",
    "step6_paired_summary.csv",
    "step6_per_attack_all_runs.csv",
    "step6_per_attack_summary.csv",
    "step6_test_curve_operating_points.csv",
    "step6_test_curve_operating_summary.csv",
    "step6_final_decision.csv",
    "step6_summary.md",
]
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in small_files:
        path = OUTPUT_ROOT / name
        if path.exists():
            zf.write(path, arcname=name)
    for dataset in DATA_PATH_CANDIDATES:
        manifest = OUTPUT_ROOT / dataset / "experiment_manifest.json"
        if manifest.exists():
            zf.write(manifest, arcname=f"manifests/{dataset}_experiment_manifest.json")
        for model in PUBLIC_MODELS:
            for seed in MODEL_SEEDS:
                task_dir = OUTPUT_ROOT / dataset / PROTOCOL_NAME / model / f"seed_{seed}"
                for filename in ["config.json", "test_metrics.json", "completed.json", "history.csv"]:
                    path = task_dir / filename
                    if path.exists():
                        zf.write(path, arcname=f"tasks/{dataset}/{model}/seed_{seed}/{filename}")

print("Decision file:", OUTPUT_ROOT / "step6_final_decision.csv")
print("Summary:", summary_path)
print("Result bundle:", bundle_path)
display(decision_df)


Decision file: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step6_public_baselines_v1/step6_final_decision.csv
Summary: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step6_public_baselines_v1/step6_summary.md
Result bundle: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step6_public_baselines_v1/THAT_IDS_step6_public_baseline_results_bundle.zip


,dataset,public_tasks_complete,public_models_available,decision,paper_message
0,CIC_V2,True,"egraphsage_public,graphids_public",PUBLIC_BASELINE_COMPARISON_COMPLETE,"Full vs egraphsage_public: ΔF1=+0.002476, ΔPR=..."
1,UNSW_V2,True,"egraphsage_public,graphids_public",PUBLIC_BASELINE_COMPARISON_COMPLETE,"Full vs egraphsage_public: ΔF1=-0.001189, ΔPR=..."
